# Expert validation 1: pick papers

The ranker is checked against expert raters on a small set of papers that spans its quality
range. This notebook:

1. Loads the pairwise comparisons from four non-paired ranker runs over the same paper pool, one
   per rubric: **novelty**, **significance**, **soundness** and **default** (overall).
2. Computes each paper's **win rate** per rubric, and a **naive overall** score (the mean of the
   novelty, significance and soundness win rates).
3. Joins the win rates with the paper metadata (internal metrics, publication status, meta-review
   score).
4. Selects 12 papers by default win rate: the 4 best, the 4 worst and the 4 closest to the median.
   `SELECTION_SET_INDEX` picks the 1st, 2nd, ... non-overlapping set.
5. Writes the table and the selection to `outputs/expert_validation/`.

## Ranker runs

Run the non-paired ranker once per rubric over the snapshots to sample from. This calls your model
endpoint (see the README).

```bash
for rubric in default novelty significance soundness; do
  python -m ranker.run_sample_and_rank_non_paired \
    "ranker_cfg@ranker=${rubric}" \
    'paper_pool.snapshot_dirs=["snapshots/<run_1>","snapshots/<run_2>"]' \
    "hydra.run.dir=outputs/expert_validation/ranker/${rubric}"
done
```

Keep the pool and sampling settings the same for all four runs. The default sampling,
`sampling.mode=all`, compares every pair of papers once.

## Setup & Configuration

In [ ]:
import json
import pathlib

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

REPO_ROOT = pathlib.Path("../..").resolve()
OUT_DIR = REPO_ROOT / "outputs/expert_validation"

# One non-paired ranker run per rubric, all over the same paper pool (see above).
RUN_DIRS = {
    dim: OUT_DIR / "ranker" / dim for dim in ["novelty", "significance", "soundness", "default"]
}

# The four runs share the paper pool; the default run's paper_data.json is the metadata source.
PAPER_DATA_PATH = RUN_DIRS["default"] / "papers" / "paper_data.json"

# ---------------------------------------------------------------------
# Selection configuration
# ---------------------------------------------------------------------
# One set of SELECTION_N_BEST + SELECTION_N_MIDDLE + SELECTION_N_WORST papers, chosen by
# SELECTION_SCORE_COL. SELECTION_SET_INDEX = n selects sets 1 to n-1 first and removes them from
# the pool, so set n shares no paper with them.
SELECTION_SET_INDEX = 2
SELECTION_SCORE_COL = "wr_default"
SELECTION_N_BEST = 4
SELECTION_N_MIDDLE = 4
SELECTION_N_WORST = 4
assert SELECTION_SET_INDEX >= 1, "SELECTION_SET_INDEX must be >= 1"

print("Repo root:", REPO_ROOT)
for dim, d in RUN_DIRS.items():
    assert (d / "comparisons.json").exists(), f"Missing comparisons.json for {dim} in {d}"
    print(f"  {dim}: {d.relative_to(REPO_ROOT)}")
assert PAPER_DATA_PATH.exists(), f"Missing {PAPER_DATA_PATH}"
print(f"  paper_data: {PAPER_DATA_PATH.relative_to(REPO_ROOT)}")

print()
print("Selection config:")
print(f"  set_index={SELECTION_SET_INDEX}")
print(f"  score_col={SELECTION_SCORE_COL}")
print(f"  n_best={SELECTION_N_BEST}, n_middle={SELECTION_N_MIDDLE}, n_worst={SELECTION_N_WORST}")

---
## Part 1: Win-Rate Computation

### Load comparisons for all dimensions

In [ ]:
comparisons = {}
for dim, run_dir in RUN_DIRS.items():
    df = pd.read_json(run_dir / "comparisons.json")

    # Build a unique paper UID across multi-snapshot pools.
    df["paper_a_uid"] = (
        df["paper_a_source_snapshot_dir"].astype(str) + "::" + df["paper_a_id"].astype(str)
    )
    df["paper_b_uid"] = (
        df["paper_b_source_snapshot_dir"].astype(str) + "::" + df["paper_b_id"].astype(str)
    )

    comparisons[dim] = df
    print(
        f"{dim}: {len(df)} comparisons, "
        f"{df['status'].value_counts().to_dict()}, "
        f"papers(unique uid): {len(set(df['paper_a_uid']) | set(df['paper_b_uid']))}"
    )

### Compute win-rate per paper

For each comparison:
- `score == 1` means position **A** won
- `score == 0` means position **B** won

Since paper order within each pair is randomized, a paper can appear in either position A or B.
We count wins and total appearances for each paper across both positions.

In [ ]:
def compute_win_rates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Given a comparisons DataFrame, compute per-paper win rate using unique paper UID.

    Returns DataFrame indexed by paper_uid with columns:
      - wins, losses, total, win_rate
    """
    # Only consider completed comparisons
    df = df[df["status"] == "completed"].copy()

    # Count wins when paper is in position A (score == 1 means A wins)
    wins_as_a = df[df["score"] == 1].groupby("paper_a_uid").size().rename("wins_as_a")

    # Count wins when paper is in position B (score == 0 means B wins)
    wins_as_b = df[df["score"] == 0].groupby("paper_b_uid").size().rename("wins_as_b")

    # Count total appearances in each position
    total_as_a = df.groupby("paper_a_uid").size().rename("total_as_a")
    total_as_b = df.groupby("paper_b_uid").size().rename("total_as_b")

    # Combine: use paper_uid as the common index
    stats = (
        pd.DataFrame(
            {
                "wins_as_a": wins_as_a,
                "wins_as_b": wins_as_b,
                "total_as_a": total_as_a,
                "total_as_b": total_as_b,
            }
        )
        .fillna(0)
        .astype(int)
    )

    stats.index.name = "paper_uid"
    stats["wins"] = stats["wins_as_a"] + stats["wins_as_b"]
    stats["total"] = stats["total_as_a"] + stats["total_as_b"]
    stats["losses"] = stats["total"] - stats["wins"]
    stats["win_rate"] = stats["wins"] / stats["total"]

    return stats[["wins", "losses", "total", "win_rate"]]

In [ ]:
win_rates = {}
for dim, df in comparisons.items():
    wr = compute_win_rates(df)
    win_rates[dim] = wr
    print(f"\n=== {dim.upper()} ===")
    print(f"Papers: {len(wr)}")
    print("Win-rate distribution:")
    print(wr["win_rate"].describe().to_string())

    # Coverage sanity: in a complete all-vs-all run each paper appears n-1 times.
    # Here we allow shortfalls due to failed comparisons.
    expected_total = len(wr) - 1
    min_total = int(wr["total"].min())
    max_total = int(wr["total"].max())
    assert max_total <= expected_total, (
        f"Found paper coverage above expected n-1 ({expected_total}). "
        f"This suggests duplicate-key issues in UID construction."
    )

    n_full = int((wr["total"] == expected_total).sum())
    n_missing = int((wr["total"] < expected_total).sum())
    print(
        f"Coverage: expected={expected_total}, range=[{min_total}, {max_total}], "
        f"full={n_full}, missing_due_to_failures={n_missing}"
    )

### Merge win-rates across dimensions

In [ ]:
# Start with paper_uid as the base
all_paper_uids = win_rates["default"].index

df_wr = pd.DataFrame(index=all_paper_uids)
df_wr.index.name = "paper_uid"

for dim in ["novelty", "significance", "soundness", "default"]:
    df_wr[f"wr_{dim}"] = win_rates[dim]["win_rate"]
    df_wr[f"n_completed_{dim}"] = win_rates[dim]["total"]

# Naive-overall: mean of novelty, significance, soundness
df_wr["wr_naive_overall"] = df_wr[["wr_novelty", "wr_significance", "wr_soundness"]].mean(axis=1)

print(f"Merged win-rates for {len(df_wr)} papers.")
print()
display(df_wr.describe())

### Load paper metadata

In [ ]:
with open(PAPER_DATA_PATH) as f:
    paper_data_raw = json.load(f)

print(f"Loaded {len(paper_data_raw)} papers from paper_data.json")
print(f"Keys per paper: {list(paper_data_raw[0].keys())}")

In [ ]:
def extract_paper_metadata(paper_data_raw: list) -> pd.DataFrame:
    """
    Extract key metadata fields from paper_data.json into a flat DataFrame.
    Uses paper_uid = source_snapshot_dir + "::" + paper_id for uniqueness.
    """
    records = []
    for p in paper_data_raw:
        rec = {
            "paper_uid": f"{p.get('source_snapshot_dir', '')}::{p['paper_id']}",
            "paper_id": p["paper_id"],
            "source_snapshot_dir": p.get("source_snapshot_dir", ""),
            "source_snapshot": p.get("source_snapshot", ""),
            "title": p["title"],
            "primary_agent_id": p["primary_agent_id"],
            "publication_t": p["publication_t"],
            "publication_status": p["publication_status"],
            "citation_count": p["citation_count"],
            "metareview_overall_score": p.get("metareview", {}).get("overall_score"),
            "metareview_decision": p.get("metareview", {}).get("decision"),
        }

        # Flatten metrics (support, soundness, significance, originality)
        metrics = p.get("metrics", {})
        if isinstance(metrics, dict):
            for category, sub_metrics in metrics.items():
                if isinstance(sub_metrics, dict):
                    for key, val in sub_metrics.items():
                        # Skip list-valued metrics (not useful as scalars)
                        if not isinstance(val, list):
                            rec[f"m_{category}_{key}"] = val

        records.append(rec)

    return pd.DataFrame(records).set_index("paper_uid")


df_meta = extract_paper_metadata(paper_data_raw)
print(f"Metadata columns ({len(df_meta.columns)}):")
for col in df_meta.columns:
    print(f"  {col}: {df_meta[col].dtype}")
print()
display(df_meta.head(3))

### Combine win-rates with metadata

In [ ]:
df_full = df_wr.join(df_meta, how="left")

# Reorder: put identifying info first, then win-rates/coverage, then metrics
id_cols = [
    "paper_id",
    "source_snapshot",
    "source_snapshot_dir",
    "title",
    "primary_agent_id",
    "publication_t",
    "publication_status",
    "citation_count",
    "metareview_overall_score",
    "metareview_decision",
]
wr_cols = [c for c in df_full.columns if c.startswith("wr_") or c.startswith("n_completed_")]
metric_cols = [c for c in df_full.columns if c.startswith("m_")]
df_full = df_full[id_cols + wr_cols + metric_cols]

print(f"Full dataframe: {df_full.shape[0]} papers x {df_full.shape[1]} columns")
print()
display(df_full.head())

In [ ]:
print("Win-rate summary statistics:")
display(df_full[wr_cols].describe())

print("\nCorrelation between win-rate dimensions:")
display(df_full[wr_cols].corr().round(3))

### Export full dataframe

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_FULL_PATH = OUT_DIR / "win_rates.json"

df_full.reset_index().to_json(EXPORT_FULL_PATH, orient="records", indent=2)
print(f"Exported full dataframe to: {EXPORT_FULL_PATH.relative_to(REPO_ROOT)}")
print(f"  {df_full.shape[0]} papers, {df_full.shape[1]} columns")

---
## Part 2: Paper Selection for Expert Rating

We select **one set of 12 papers** using the **default** (overall) win-rate.

Papers with `_incomplete` in their ID are excluded: their research session ended before the paper
was finished, so they should not be given to raters.

The set contains:
- **4 best** (highest default win-rate)
- **4 worst** (lowest default win-rate)
- **4 middle** (of the remaining papers, those closest to their median default win-rate)

For `SELECTION_SET_INDEX = n`, sets 1 to n-1 are selected the same way and removed from the pool
first, so the sets never overlap and each samples the full quality range.

In [ ]:
# Filter out _incomplete papers (sessions that ended before the paper was finished)
incomplete_mask = df_full.index.str.contains("_incomplete")
df_eligible = df_full[~incomplete_mask].copy()
n_excluded = int(incomplete_mask.sum())
print(f"Excluded {n_excluded} incomplete papers: {df_full.index[incomplete_mask].tolist()}")
print(f"Eligible for selection: {len(df_eligible)} papers")
print()


def select_single_paper_set_from_pool(
    df: pd.DataFrame,
    score_col: str,
    n_best: int,
    n_worst: int,
    n_middle: int,
) -> list[str]:
    """
    Select one set from the provided pool: n_best BEST, n_middle MIDDLE, n_worst WORST.
    """
    target = n_best + n_middle + n_worst
    if len(df) < target:
        raise ValueError(f"Need at least {target} papers for single-set selection, got {len(df)}.")

    sorted_df = df.sort_values(score_col, ascending=False)
    all_ids = sorted_df.index.tolist()

    best_ids = all_ids[:n_best]
    worst_ids = all_ids[-n_worst:]

    used = set(best_ids + worst_ids)
    remaining = sorted_df.drop(index=list(used)).copy()
    median_val = remaining[score_col].median()
    remaining["dist_to_median"] = (remaining[score_col] - median_val).abs()

    middle_ids = (
        remaining.sort_values(["dist_to_median", score_col], ascending=[True, False])
        .head(n_middle)
        .index.tolist()
    )

    set_ids = best_ids + middle_ids + worst_ids

    assert len(set(set_ids)) == target, "Single set contains duplicates"
    assert len(set_ids) == target
    return set_ids


def select_nth_single_paper_set(
    df: pd.DataFrame,
    score_col: str,
    n_best: int,
    n_worst: int,
    n_middle: int,
    set_index: int,
) -> tuple[list[str], list[list[str]]]:
    """
    Select the 1st/2nd/nth single set by iteratively removing prior selected sets.

    Returns:
      - selected_set: the requested nth set
      - prior_sets: list of previously selected sets (length set_index - 1)
    """
    if set_index < 1:
        raise ValueError(f"set_index must be >= 1, got {set_index}")

    per_set = n_best + n_middle + n_worst
    required_total = set_index * per_set
    if len(df) < required_total:
        raise ValueError(
            f"Need at least {required_total} eligible papers to select set_index={set_index}, got {len(df)}."
        )

    remaining = df.copy()
    prior_sets: list[list[str]] = []

    for i in range(1, set_index + 1):
        current_set = select_single_paper_set_from_pool(
            remaining,
            score_col=score_col,
            n_best=n_best,
            n_worst=n_worst,
            n_middle=n_middle,
        )

        if i < set_index:
            prior_sets.append(current_set)
            remaining = remaining.drop(index=current_set)
        else:
            return current_set, prior_sets

    raise RuntimeError("Unreachable")


selected_set_label = f"set_{SELECTION_SET_INDEX}"
selected_set, prior_sets = select_nth_single_paper_set(
    df_eligible,
    score_col=SELECTION_SCORE_COL,
    n_best=SELECTION_N_BEST,
    n_worst=SELECTION_N_WORST,
    n_middle=SELECTION_N_MIDDLE,
    set_index=SELECTION_SET_INDEX,
)

print(f"Selected set label: {selected_set_label}")
print(f"Excluded prior sets: {len(prior_sets)}")
print(f"Selected: {len(selected_set)} papers")

# Verify no incomplete papers slipped through
incomplete_in_set = [p for p in selected_set if "_incomplete" in p]
assert len(incomplete_in_set) == 0, f"Selection contains incomplete papers: {incomplete_in_set}"
print("Verified: no incomplete papers in the selected set.")

### Display selected papers with quality statistics

In [ ]:
DISPLAY_COLS = [
    "title",
    "publication_status",
    "publication_t",
    "metareview_overall_score",
    "wr_default",
    "wr_novelty",
    "wr_significance",
    "wr_soundness",
    "wr_naive_overall",
    "m_support_external_papers_read",
    "m_support_successful_code_executions",
    "m_support_lines_of_code",
    "m_support_analytical_depth",
    "m_support_claim_evidence_alignment_max_cosine",
    "m_soundness_internal_consistency_score",
    "m_soundness_diagnostic_validity_pass_rate",
    "m_significance_clinical_biological_relevance",
    "m_originality_semantic_novelty_knn_distance",
    "m_originality_embedding_novelty_percentile",
    "citation_count",
]


def display_paper_set(paper_ids: list[str], set_name: str, df: pd.DataFrame):
    """Display detailed quality stats for a set of papers."""
    print()
    print("=" * 100)
    print(f"  {set_name}")
    print("=" * 100)

    if not paper_ids:
        print("  (empty set)")
        return

    subset = df.loc[paper_ids]

    # Categorize papers by their role in the set
    sorted_by_wr = subset.sort_values("wr_default", ascending=False)
    ids_sorted = sorted_by_wr.index.tolist()

    # Determine categories based on wr_default rank within the full dataset
    full_rank = df["wr_default"].rank(ascending=False)

    for pid in ids_sorted:
        rank = int(full_rank[pid])
        row = df.loc[pid]

        # Determine category
        if row["wr_default"] >= subset["wr_default"].quantile(0.75):
            cat = "BEST"
        elif row["wr_default"] <= subset["wr_default"].quantile(0.25):
            cat = "WORST"
        else:
            cat = "MIDDLE"

        print()
        print(f"--- [{cat}] {pid} (overall rank: {rank}/{len(df)}) ---")
        print(f"  Title: {row['title'][:100]}")
        print(f"  Publication: status={row['publication_status']}, t={row['publication_t']}")
        print(f"  Metareview: overall_score={row['metareview_overall_score']}")
        print(
            f"  Win-rates: default={row['wr_default']:.3f}, novelty={row['wr_novelty']:.3f}, "
            f"significance={row['wr_significance']:.3f}, soundness={row['wr_soundness']:.3f}"
        )
        print(f"  Naive-overall: {row['wr_naive_overall']:.3f}")
        print(
            f"  Support: papers_read={row['m_support_external_papers_read']}, "
            f"code_execs={row['m_support_successful_code_executions']}, "
            f"loc={row['m_support_lines_of_code']}, "
            f"analytical_depth={row['m_support_analytical_depth']}"
        )
        print(
            f"  Soundness: consistency={row['m_soundness_internal_consistency_score']}, "
            f"diagnostic_validity={row['m_soundness_diagnostic_validity_pass_rate']}"
        )
        print(
            f"  Significance: clinical_bio_relevance={row['m_significance_clinical_biological_relevance']}"
        )
        print(
            f"  Originality: semantic_novelty_knn={row['m_originality_semantic_novelty_knn_distance']:.3f}, "
            f"novelty_percentile={row['m_originality_embedding_novelty_percentile']}"
        )
        print(f"  Citations: {row['citation_count']}")


display_paper_set(
    selected_set, f"{selected_set_label.upper()} ({len(selected_set)} papers)", df_full
)

### Compact table view of selected papers

In [ ]:
available_display_cols = [c for c in DISPLAY_COLS if c in df_full.columns]

print(f"=== {selected_set_label.upper()} ===")
display(
    df_full.loc[selected_set, available_display_cols].sort_values("wr_default", ascending=False)
)

### Export selection

In [ ]:
# --- Export 1: ID list and settings (read by notebook 02) ---
EXPORT_LIST_PATH = OUT_DIR / "selection_list.json"
selection_sets_for_export = {selected_set_label: selected_set}

selection_list = {
    "selection_set_index": SELECTION_SET_INDEX,
    "selection_set_label": selected_set_label,
    "score_col": SELECTION_SCORE_COL,
    "n_best": SELECTION_N_BEST,
    "n_middle": SELECTION_N_MIDDLE,
    "n_worst": SELECTION_N_WORST,
    "paper_data_path": str(PAPER_DATA_PATH.relative_to(REPO_ROOT)),
    "selection_sets": selection_sets_for_export,
}

with open(EXPORT_LIST_PATH, "w") as f:
    json.dump(selection_list, f, indent=2)

print(f"Exported selection list to: {EXPORT_LIST_PATH.relative_to(REPO_ROOT)}")
print(f"  {selected_set_label} ({len(selected_set)}): {selected_set}")

# --- Export 2: Full data for selected papers with set & category labels (read by notebook 03) ---
EXPORT_SELECTION_PATH = OUT_DIR / "selection.json"


def build_selection_df(
    df: pd.DataFrame,
    selection_sets: dict[str, list[str]],
    score_col: str = "wr_default",
    n_best: int = 4,
) -> pd.DataFrame:
    """
    Build a DataFrame of selected papers with 'set' and 'category' columns.

    Category assignment per set: sorted by score_col.
    Top n_best = BEST, bottom n_best = WORST, middle = MIDDLE.
    """
    rows = []
    for set_label, paper_ids in selection_sets.items():
        if not paper_ids:
            continue

        subset = df.loc[paper_ids].sort_values(score_col, ascending=False)
        ordered_ids = subset.index.tolist()
        for i, pid in enumerate(ordered_ids):
            if i < n_best:
                category = "BEST"
            elif i >= len(ordered_ids) - n_best:
                category = "WORST"
            else:
                category = "MIDDLE"
            row = df.loc[pid].copy()
            row["set"] = set_label
            row["category"] = category
            row.name = pid
            rows.append(row)

    return pd.DataFrame(rows)


df_selection = build_selection_df(
    df_full,
    selection_sets_for_export,
    score_col=SELECTION_SCORE_COL,
    n_best=SELECTION_N_BEST,
)

# Put set & category columns first
label_cols = ["set", "category"]
other_cols = [c for c in df_selection.columns if c not in label_cols]
df_selection = df_selection[label_cols + other_cols]

df_selection.reset_index().to_json(EXPORT_SELECTION_PATH, orient="records", indent=2)

print()
print(f"Exported selection data to: {EXPORT_SELECTION_PATH.relative_to(REPO_ROOT)}")
print(f"  {len(df_selection)} papers, {len(df_selection.columns)} columns")
print()
print("Breakdown:")
print(df_selection.groupby(["set", "category"]).size().unstack(fill_value=0))

---
## Summary

### Outputs (in `outputs/expert_validation/`)
- `win_rates.json`: every paper, with its win rates for the four rubrics, the naive-overall score
  and the paper metadata.
- `selection.json`: the selected papers' rows, with `set` and `category` (BEST/MIDDLE/WORST)
  labels. Notebook 03 reads it.
- `selection_list.json`: the selected paper IDs, the selection settings and the path of the
  ranker's `paper_data.json`. Notebook 02 reads it.